In [17]:
import json
import os
from dotenv import load_dotenv
from langchain.schema import Document
import pandas as pd
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.chains import ConversationalRetrievalChain
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load biến môi trường từ file .env
load_dotenv(r"D:\AI\.env")
print("GOOGLE_API_KEY present:", bool(os.getenv("GOOGLE_API_KEY")))

GOOGLE_API_KEY present: True


In [4]:
# 1. Load dữ liệu từ file JSON
with open("D:\\AI\\assistant\\data\\dulieu.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [5]:
documents = []
for item in data:
    for chunk in item["chunks"]:
        # Mỗi chunk là một Document, kèm metadata (nguồn, tiêu đề)
        doc = Document(
            page_content=chunk,
            metadata={"source": item["url"], "title": item["title"]}
        )
        documents.append(doc)

print(f"Tổng số chunks cần index: {len(documents)}")

Tổng số chunks cần index: 38


In [6]:
# 3. Tạo Vector Database
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vectorstore = FAISS.from_documents(documents, embeddings)

c:\Users\Admin\miniconda3\envs\KhoaHocT3H\Lib\site-packages\langchain_core\_api\deprecation.py:139: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(


In [7]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 6}
)

In [8]:
from langchain.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Bạn là một chatbot thời sự chuyên nghiệp.

Nhiệm vụ của bạn là trả lời câu hỏi CHỈ dựa trên thông tin trong context.
KHÔNG suy đoán, KHÔNG bịa.

Nếu không có thông tin phù hợp, hãy trả lời:
"Hiện tại tôi chưa tìm thấy thông tin phù hợp trong dữ liệu."

Context:
{context}

Câu hỏi:
{question}

Trả lời (tiếng Việt, ngắn gọn, trung lập):
""")


In [18]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
    max_output_tokens=1024
    
)

In [19]:
from langchain.chains import ConversationalRetrievalChain

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    combine_docs_chain_kwargs={"prompt": prompt},
    return_source_documents=True
)


In [20]:
chat_history = []

query = "tin tức về chuyến thăm và các hoạt động giao lưu của Phu nhân Tổng Bí thư"


result = qa_chain({
    "question": query,
    "chat_history": chat_history
})

print("TRẢ LỜI:")
print(result["answer"])


c:\Users\Admin\miniconda3\envs\KhoaHocT3H\Lib\site-packages\langchain_core\_api\deprecation.py:139: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


TRẢ LỜI:
Ngày 27/1, Phu nhân Tổng Bí thư Đảng Cộng sản Việt Nam Ngô Phương Ly và Phu nhân Tổng Bí thư, Chủ tịch nước Lào Naly Sisoulith đã đến thăm và tham gia các hoạt động tại tỉnh Ninh Bình.

Trong khuôn khổ chuyến thăm, hai Phu nhân đã:
*   Thăm, dâng hương tại Đền thờ Công chúa Nhồi Hoa.
*   Tham gia hoạt động giao lưu với các nghệ nhân tiêu biểu một số làng nghề truyền thống của tỉnh Ninh Bình tại Phòng trưng bày Khách xá Bái Đính, và nghe giới thiệu về các nghề thêu, dệt tơ tằm, đan cói và gốm Bồ Bát.
*   Phu nhân Ngô Phương Ly đã tặng Phu nhân Tổng Bí thư, Chủ tịch nước Lào chiếc khăn thêu hình hoa Chăm pa do nghệ nhân làng thêu Văn Lâm thực hiện dựa trên bản vẽ và thiết kế của bà Ngô Phương Ly.
